# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
pp = pprint.PrettyPrinter()

print(f"Dataset Name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")
print(f"License: {metadata.get('license')}")
print(f"Published: {metadata.get('datePublished')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced via their `@id` fields.

In [ ]:
# List the available record sets

# The metadata['recordSet'] holds the list of record sets if present
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print("No record sets found directly in metadata. Attempting to discover from dataset...")
    # mlcroissant exposes them as dataset.record_sets
    record_sets = dataset.record_sets

print("Available record sets (@id):")
for rs in record_sets:
    print(f"  {rs['@id']}")

# For each record set, list its fields and columns by @id
all_fields = {}
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    # The fields can be present as `fields` or similar in Croissant
    fields = rs.get('field', [])
    if not fields:
        # Try columns (if present)
        columns = rs.get('column', [])
        if columns:
            print("  Columns @id:")
            for col in columns:
                print(f"    {col['@id']}")
        else:
            print("  No fields or columns found in this record set.")
        continue
    print("  Fields @id:")
    for field in fields:
        print(f"    {field['@id']}")
        all_fields[field['@id']] = field

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record sets to extract (using their @id)
record_set_ids = []
for rs in record_sets:
    record_set_ids.append(rs['@id'])

dataframes = {}
for rec_id in record_set_ids:
    print(f"Loading records for record set {rec_id}")
    try:
        records = list(dataset.records(record_set=rec_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"  Columns ({len(df.columns)}): {df.columns.tolist()}\n")
            print(df.head(3))
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

# For demonstration, select the first available record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    df_main = dataframes.get(first_rs_id)
    if df_main is not None:
        print(f"\nColumns in DataFrame ({first_rs_id}):\n{df_main.columns.tolist()}")
        df_main.head()
    else:
        print(f"No data found for record set {first_rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All field references are made using their `@id` names.**

In [ ]:
# EDA section
# Select a numeric field (e.g., GAD-7 score, PHQ-9 score, PSQ etc.)
# We will use whatever numeric fields are present based on DataFrame columns

if df_main is not None and not df_main.empty:
    # Try common mental health fields
    possible_numeric_fields = [
        'GAD_7_score', 'PHQ_9_score', 'PSQ_score', # possible column names
        'gad_7_score', 'phq_9_score', 'psq_score', # lowercase variants
        'score_gad_7', 'score_phq_9', 'score_psq',
        # Or choose the first numeric column
    ]
    numeric_field = None
    for col in df_main.columns:
        if col in possible_numeric_fields:
            numeric_field = col
            break

    if not numeric_field:
        # Try to auto-select numeric columns
        num_cols = df_main.select_dtypes(include=['number']).columns
        if len(num_cols) > 0:
            numeric_field = num_cols[0]

    print(f"Using numeric field for analysis: {numeric_field}")
    if numeric_field:
        # Filter records with > threshold
        threshold = 10
        filtered_df = df_main[df_main[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        group_field_candidates = [
            'level_of_education', 'gender', 'village', 'marital_status',
            'Level_of_Education', 'Gender', 'Village', 'Marital_Status'
        ]
        group_field = None
        for col in df_main.columns:
            if col in group_field_candidates:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped average {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No obvious grouping field found in the dataset.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available or DataFrame is empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and numeric_field:
    plt.figure(figsize=(8,6))
    sns.histplot(df_main[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field is found, visualize group differences
    if group_field:
        plt.figure(figsize=(8,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df_main)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides survey records on mental health indicators and demographics for residents of Kilifi County, Kenya.
- Using `mlcroissant`, we loaded metadata and record sets, referencing all entities by their `@id` fields.
- Exploratory analysis and filtering demonstrated how to select and analyze mental health scores, normalize them, and group by demographic attributes (when present).
- Visualizations further helped to reveal distributions and group-level differences.
- The approach and tools shown here serve as a reproducible template for AI-ready FAIR data processing in Python.